# C4: Robustness Evaluation under Data Distribution Shift — Multi-Run Reproducibility

**MULTIRUN WRAPPER ONLY — original logic preserved.**

Curves và bars hiển thị **mean ± std over 5 independent train runs** dùng cùng 1 fixed test set.  
Toàn bộ evaluation protocol C4 giữ nguyên; chỉ training sample thay đổi qua từng run.

| Exp | Tên | Câu hỏi |
|-----|-----|---------|
| **E1** | Temporal split evaluation | QSVM có giữ được F1 trên KDDTest-21 (khó hơn KDDTest+)? |
| **E2** | Feature perturbation robustness | Degradation slope của QSVM so với SVM-RBF/Poly theo σ noise? |
| **E3** | Class prior shift | Margin của QSVM có generalize tốt khi tỷ lệ Normal/Attack thay đổi? |

## 1. Imports & Config

Thiết lập thư viện, đường dẫn, seed, danh sách 5 training runs và style/metadata dùng xuyên suốt C4 multi-run.


In [10]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os, time, json, warnings
warnings.filterwarnings('ignore')

from scipy.stats import chi2 as chi2_dist
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

import qiskit, qiskit_machine_learning
print(f'Qiskit              : {qiskit.__version__}')
print(f'Qiskit ML           : {qiskit_machine_learning.__version__}')
print('Backend             : FidelityStatevectorKernel (statevector, noiseless)')

Qiskit              : 2.3.0
Qiskit ML           : 0.9.0
Backend             : FidelityStatevectorKernel (statevector, noiseless)


## 1b. Multi-Run Configuration

Giữ nguyên logic notebook gốc, nhưng lặp qua 5 tập train độc lập và báo cáo `mean ± std` trên cùng fixed test protocol.


In [11]:
# ─── MULTIRUN WRAPPER ONLY — original logic preserved ────────────────────────
RANDOM_STATE = 42
RUN_IDS      = [1, 2, 3, 4, 5]
np.random.seed(RANDOM_STATE)

DATA_DIR   = '../data/processed_data'
MODELS_DIR = '../models'
OUTPUT_DIR = '../results/c4_multirun'
FIG_DIR    = f'{OUTPUT_DIR}/figures'
CACHE_DIR  = f'{OUTPUT_DIR}/cache'
for d in [OUTPUT_DIR, FIG_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

N_QUBITS     = 4
ANGLE_MAX    = np.pi
REPS         = 2
ENTANGLEMENT = 'full'
C_QSVM       = 1.0

# C_SVM follows the C3 notebook: one explicit value per SVM variant.
SVM_MODEL_KEYS = [
    'SVM-Linear (mm)', 'SVM-Linear (std)',
    'SVM-Poly2 (mm)',  'SVM-Poly2 (std)',
    'SVM-RBF (mm)',    'SVM-RBF (std)',
]
SVM_C_VALUES   = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 10.0], dtype=float)
SVM_C_BY_MODEL = dict(zip(SVM_MODEL_KEYS, SVM_C_VALUES))

def _fmt_c_for_tag(c):
    return f'{float(c):g}'.replace('.', 'p')

SVM_C_TAG = '-'.join(
    f'{k.replace("SVM-", "").replace(" ", "").replace("(", "").replace(")", "").lower()}{_fmt_c_for_tag(SVM_C_BY_MODEL[k])}'
    for k in SVM_MODEL_KEYS
)

TRAIN_SIZE   = 1000
SIGMA_LEVELS = [0.0, 0.01, 0.05, 0.1, 0.2]
LABEL_COLS   = ['label', 'label_binary', 'label_multiclass', 'attack_category']
CONFIG_TAG   = f'reps{REPS}_{ENTANGLEMENT}_C{C_QSVM}_cs{SVM_C_TAG}_n{TRAIN_SIZE}'

# ─── Color palette nhất quán với notebook gốc C4 ─────────────────────────────
PLOT_COLORS = {
    'QSVM (ZZ)'        : '#4C72B0',
    'SVM-RBF (mm)'     : '#DD8452',   'SVM-RBF (std)'    : '#F4A460',
    'SVM-Poly2 (mm)'   : '#55A868',   'SVM-Poly2 (std)'  : '#90EE90',
    'SVM-Linear (mm)'  : '#C44E52',   'SVM-Linear (std)' : '#F08080',
}
MODEL_ORDER = list(PLOT_COLORS.keys())
MARKERS = {
    'QSVM (ZZ)'        : 'o',
    'SVM-RBF (mm)'     : 's',    'SVM-RBF (std)'    : 's',
    'SVM-Poly2 (mm)'   : '^',    'SVM-Poly2 (std)'  : '^',
    'SVM-Linear (mm)'  : 'D',    'SVM-Linear (std)' : 'D',
}
LINESTYLES = {
    'QSVM (ZZ)'        : '-',
    'SVM-RBF (mm)'     : '--',   'SVM-RBF (std)'    : ':',
    'SVM-Poly2 (mm)'   : '-.',   'SVM-Poly2 (std)'  : ':',
    'SVM-Linear (mm)'  : '--',   'SVM-Linear (std)' : ':',
}

print(f'CONFIG_TAG : {CONFIG_TAG}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'RUN_IDS    : {RUN_IDS}')
print('SVM_C_BY_MODEL:')
for _k in SVM_MODEL_KEYS:
    print(f'  {_k:<18s}: C={SVM_C_BY_MODEL[_k]:g}')

CONFIG_TAG : reps2_full_C1.0_cslinearmm0p1-linearstd0p1-poly2mm0p1-poly2std0p1-rbfmm0p1-rbfstd10_n1000
OUTPUT_DIR : ../results/c4_multirun
RUN_IDS    : [1, 2, 3, 4, 5]
SVM_C_BY_MODEL:
  SVM-Linear (mm)   : C=0.1
  SVM-Linear (std)  : C=0.1
  SVM-Poly2 (mm)    : C=0.1
  SVM-Poly2 (std)   : C=0.1
  SVM-RBF (mm)      : C=0.1
  SVM-RBF (std)     : C=10


## 2. Load Pipeline Transformers (Zero-Leakage)

Tải `selector`, `pca`, `scaler` đã fit trước đó. Không fit lại trong C4 để bảo toàn zero-leakage contract.


In [12]:
# ─── Load transformers đã fit trên full train — KHÔNG fit lại ────────────────
selector = joblib.load(f'{MODELS_DIR}/feature_selector_k20.joblib')
pca      = joblib.load(f'{MODELS_DIR}/pca_4components.joblib')
scaler   = joblib.load(f'{MODELS_DIR}/scaler_minmax_pi.joblib')
PC_LABELS = [f'PC{i+1}' for i in range(N_QUBITS)]
print(f'SelectKBest : k={selector.k}')
print(f'PCA         : n_components={pca.n_components_}, variance={pca.explained_variance_ratio_.sum()*100:.2f}%')


def transform_pipeline(df, feature_cols):
    """Raw → SelectKBest → PCA → MinMaxScaler [0,π]. KHÔNG fit lại."""
    X_raw = df[feature_cols].to_numpy(dtype=np.float32)
    X_sel = selector.transform(X_raw)
    X_pca = pca.transform(X_sel)
    return np.clip(scaler.transform(X_pca), 0, ANGLE_MAX).astype(np.float64)


def pca_only(df, feature_cols):
    """Raw → SelectKBest → PCA (trước MinMax). Dùng cho std branch."""
    X_raw = df[feature_cols].to_numpy(dtype=np.float32)
    X_sel = selector.transform(X_raw)
    return pca.transform(X_sel).astype(np.float64)


def build_qsvm(C=C_QSVM):
    fm = zz_feature_map(feature_dimension=N_QUBITS, reps=REPS, entanglement=ENTANGLEMENT)
    kernel = FidelityStatevectorKernel(feature_map=fm, shots=None, enforce_psd=True, cache_size=None)
    return SVC(kernel=kernel.evaluate, C=C, probability=True, random_state=RANDOM_STATE)


def _allocate_exact_counts(group_sizes, n_total, min_counts=None):
    """Allocate integer counts that sum exactly to n_total."""
    keys = list(group_sizes.keys())
    min_counts = min_counts or {}
    counts = {k: int(min_counts.get(k, 0)) for k in keys}
    remaining = int(n_total) - sum(counts.values())
    if remaining < 0:
        scale = n_total / max(sum(counts.values()), 1)
        raw = {k: counts[k] * scale for k in keys}
    else:
        weights = {k: max(int(group_sizes[k]) - counts[k], 0) for k in keys}
        total_w = sum(weights.values())
        raw = {k: counts[k] + remaining * weights[k] / total_w for k in keys} if total_w else {k: counts[k] for k in keys}
    base = {k: int(np.floor(raw[k])) for k in keys}
    remainder = int(n_total) - sum(base.values())
    frac_order = sorted(keys, key=lambda k: (raw[k] - base[k], group_sizes[k]), reverse=True)
    for k in frac_order[:max(remainder, 0)]:
        base[k] += 1
    if remainder < 0:
        for k in sorted(keys, key=lambda kk: (base[kk], group_sizes[kk]), reverse=True)[:abs(remainder)]:
            base[k] -= 1
    return base


def stratified_sample_for_qsvm(df, n_samples=1000, min_rare=30,
                                rare_categories=('U2R', 'R2L'), random_state=42,
                                stratify_col='attack_category'):
    """Stratified sample with exactly n_samples rows, preserving rare attacks when possible."""
    if len(df) == 0:
        raise ValueError('Cannot sample from an empty dataframe')
    rng = np.random.RandomState(random_state)
    groups = {cat: len(pool) for cat, pool in df.groupby(stratify_col, sort=False)}
    rare_cats = [c for c in rare_categories if c in groups]
    min_counts = {c: min(int(min_rare), int(n_samples), groups[c]) for c in rare_cats}
    counts = _allocate_exact_counts(groups, int(n_samples), min_counts=min_counts)
    parts = []
    for cat, n_take in counts.items():
        if n_take <= 0:
            continue
        pool = df[df[stratify_col] == cat]
        parts.append(pool.sample(n=n_take, replace=len(pool) < n_take,
                                 random_state=rng.randint(1_000_000)))
    result = pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    assert len(result) == n_samples, f'Expected {n_samples}, got {len(result)}'
    return result


def sample_binary_exact(df, n_normal, n_attack, random_state=42):
    """Exact binary sample by label_binary, using replacement only if a class is short."""
    rng = np.random.RandomState(random_state)
    parts = []
    for label, n_take in [(0, int(n_normal)), (1, int(n_attack))]:
        pool = df[df['label_binary'] == label]
        if len(pool) == 0:
            raise ValueError(f'Missing label_binary={label} for exact sampling')
        parts.append(pool.sample(n=n_take, replace=len(pool) < n_take,
                                 random_state=rng.randint(1_000_000)))
    return pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)


def sample_attack_category_exact(df, category_counts, random_state=42):
    """Exact sample by attack_category counts."""
    rng = np.random.RandomState(random_state)
    parts = []
    for cat, n_take in category_counts.items():
        pool = df[df['attack_category'] == cat]
        if len(pool) == 0:
            raise ValueError(f'Missing attack_category={cat} for exact sampling')
        parts.append(pool.sample(n=int(n_take), replace=len(pool) < int(n_take),
                                 random_state=rng.randint(1_000_000)))
    result = pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    assert len(result) == sum(category_counts.values())
    return result

def mcnemar_test(y_true, pred_a, pred_b):
    """McNemar test với continuity correction (Edwards)."""
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    n01 = np.sum(correct_a & ~correct_b)
    n10 = np.sum(~correct_a & correct_b)
    n_disc = n01 + n10
    if n_disc == 0:
        return 0.0, 1.0, 0
    chi2 = (abs(n01 - n10) - 1)**2 / (n01 + n10)
    return chi2, 1 - chi2_dist.cdf(chi2, df=1), n_disc

SelectKBest : k=20
PCA         : n_components=4, variance=86.62%


## 3. Fixed Test Sets & Shared Evaluation Data

Chuẩn bị các test set dùng chung cho mọi run: temporal split, perturbation robustness và class-prior shift.


In [13]:
# ─── Fixed test sets — dùng chung cho tất cả 5 runs ──────────────────────────
probe        = pd.read_csv(f'{DATA_DIR}/multi_run/train_run1.csv')
feature_cols = [c for c in probe.columns if c not in LABEL_COLS]

assert len(probe) == TRAIN_SIZE, f'train_run1 expected {TRAIN_SIZE}, got {len(probe)}'

df_test_std = pd.read_csv(f'{DATA_DIR}/NSL_KDD_Test_Cleaned.csv')
df_test_21  = pd.read_csv(f'{DATA_DIR}/NSL_KDD_Test21_Cleaned.csv')
print(f'KDDTest+   (standard) : {df_test_std.shape}')
print(f'KDDTest-21 (hard)     : {df_test_21.shape}')


def _load_or_build_exact(path, builder_fn, expected_n, name):
    if os.path.exists(path):
        df = pd.read_csv(path)
        if len(df) == expected_n:
            return df
        print(f'[REBUILD] {name}: cached size={len(df)} != expected={expected_n}')
    df = builder_fn()
    assert len(df) == expected_n, f'{name}: expected {expected_n}, got {len(df)}'
    try:
        df.to_csv(path, index=False)
    except PermissionError:
        stem, ext = os.path.splitext(path)
        alt_path = f'{stem}_exact{ext}'
        df.to_csv(alt_path, index=False)
        print(f'[WARN] Cannot overwrite locked cache: {path}')
        print(f'[SAVED] exact-size cache instead: {alt_path}')
    return df

# ── E1 fixed samples ──────────────────────────────────────────────────────────
E1_N = 500; E1_MIN_RARE = max(5, E1_N // 25)
e1_std_path = f'{DATA_DIR}/NSL_KDD_Test_C4_E1_Std_Sample{E1_N}.csv'
e1_21_path  = f'{DATA_DIR}/NSL_KDD_Test_C4_E1_Hard_Sample{E1_N}.csv'

df_e1_std = _load_or_build_exact(
    e1_std_path,
    lambda: stratified_sample_for_qsvm(df_test_std, E1_N, E1_MIN_RARE, random_state=RANDOM_STATE),
    E1_N,
    'E1 KDDTest+'
)
df_e1_21 = _load_or_build_exact(
    e1_21_path,
    lambda: stratified_sample_for_qsvm(df_test_21, E1_N, E1_MIN_RARE, random_state=RANDOM_STATE),
    E1_N,
    'E1 KDDTest-21'
)

# ── E2 fixed sample ───────────────────────────────────────────────────────────
E2_N = 400; E2_MIN_RARE = max(5, E2_N // 25)
e2_base_path = f'{DATA_DIR}/NSL_KDD_Test_C4_E2_Sample{E2_N}.csv'
df_e2_base = _load_or_build_exact(
    e2_base_path,
    lambda: stratified_sample_for_qsvm(df_test_std, E2_N, E2_MIN_RARE, random_state=RANDOM_STATE + 10),
    E2_N,
    'E2 base'
)

# ── E3 fixed samples: 3 distributions ────────────────────────────────────────
E3_N = 300; E3_MIN_RARE = max(5, E3_N // 25)
df_test_all = df_test_std.copy()
df_normal   = df_test_all[df_test_all['label_binary'] == 0]
df_attack   = df_test_all[df_test_all['label_binary'] == 1]

e3a_path = f'{DATA_DIR}/NSL_KDD_Test_C4_E3a_Balanced_Sample{E3_N}.csv'
e3b_path = f'{DATA_DIR}/NSL_KDD_Test_C4_E3b_AttackHeavy_Sample{E3_N}.csv'
e3c_path = f'{DATA_DIR}/NSL_KDD_Test_C4_E3c_DoSOnly_Sample{E3_N}.csv'

def build_e3a():
    return sample_binary_exact(df_test_all, E3_N // 2, E3_N - E3_N // 2,
                               random_state=RANDOM_STATE + 200)

def build_e3b():
    n_atk = int(round(E3_N * 0.7)); n_nrm = E3_N - n_atk
    normal_part = df_normal.sample(n=n_nrm, replace=len(df_normal) < n_nrm,
                                   random_state=RANDOM_STATE + 201)
    attack_part = stratified_sample_for_qsvm(df_attack, n_atk, E3_MIN_RARE,
                                             random_state=RANDOM_STATE + 201)
    result = pd.concat([normal_part, attack_part]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    assert len(result) == E3_N
    return result

def build_e3c():
    return sample_attack_category_exact(
        df_test_all,
        {'DoS': E3_N // 2, 'Normal': E3_N - E3_N // 2},
        random_state=RANDOM_STATE + 202
    )

df_e3a = _load_or_build_exact(e3a_path, build_e3a, E3_N, 'E3 balanced')
df_e3b = _load_or_build_exact(e3b_path, build_e3b, E3_N, 'E3 attack-heavy')
df_e3c = _load_or_build_exact(e3c_path, build_e3c, E3_N, 'E3 DoS-only')

test_sets_e3 = {
    '(a) Balanced\n50-50'   : df_e3a,
    '(b) Attack-heavy\n70%' : df_e3b,
    '(c) DoS-only\nbinary'  : df_e3c,
}
E3_SET_NAMES_DISPLAY = list(test_sets_e3.keys())  # với newline cho axis label

print('\nFixed test sets:')
for name, df in test_sets_e3.items():
    assert len(df) == E3_N, f'{name}: expected {E3_N}, got {len(df)}'
    dist = df['label_binary'].value_counts(); pct = dist / dist.sum() * 100
    print(f'  {name.replace(chr(10), " "):30s}: {len(df)} mẫu | Normal={pct.get(0,0):.1f}%  Attack={pct.get(1,0):.1f}%')
assert len(df_e1_std) == E1_N and len(df_e1_21) == E1_N
assert len(df_e2_base) == E2_N
print(f'\n[E1] std={len(df_e1_std)} hard={len(df_e1_21)}')
print(f'[E2] base={len(df_e2_base)}')
print('Fixed test sets ready.')


KDDTest+   (standard) : (22544, 126)
KDDTest-21 (hard)     : (11850, 126)

Fixed test sets:
  (a) Balanced 50-50            : 300 mẫu | Normal=50.0%  Attack=50.0%
  (b) Attack-heavy 70%          : 300 mẫu | Normal=30.0%  Attack=70.0%
  (c) DoS-only binary           : 300 mẫu | Normal=50.0%  Attack=50.0%

[E1] std=500 hard=500
[E2] base=400
Fixed test sets ready.


## 4. Single-Run C4 Wrapper (Original Logic Preserved)

Hàm này train/load model cho một `train_run{i}` rồi chạy đầy đủ ba experiment C4: temporal shift, feature perturbation và class prior shift.


In [14]:
def run_c4_single_train(train_path, run_id):
    """MULTIRUN WRAPPER ONLY — original single-run C4 logic preserved."""
    print(f'\n[C4][run_{run_id}] start → {train_path}')
    run_cache = f'{CACHE_DIR}/run_{run_id}'
    os.makedirs(run_cache, exist_ok=True)

    train_df = pd.read_csv(train_path)
    assert len(train_df) == TRAIN_SIZE, f'run {run_id}: expected {TRAIN_SIZE}, got {len(train_df)}'
    assert feature_cols == [c for c in train_df.columns if c not in LABEL_COLS]

    X_train     = transform_pipeline(train_df, feature_cols)
    X_train_pca = pca_only(train_df, feature_cols)
    y_train     = train_df['label_binary'].to_numpy(dtype=np.int64)
    std_scaler  = StandardScaler()
    X_train_std = std_scaler.fit_transform(X_train_pca)

    # ── Train / load cached models ────────────────────────────────────────────
    model_path = f'{run_cache}/models_{CONFIG_TAG}_run{run_id}.joblib'
    if os.path.exists(model_path):
        store      = joblib.load(model_path)
        CLFS       = store['classifiers']
        std_scaler = store['std_scaler']
        print(f'[C4][run_{run_id}] cache hit models')
    else:
        CLFS = {}
        t0 = time.time()
        print(f'  1/7  QSVM (ZZFeatureMap) ...')
        CLFS['QSVM (ZZ)'] = build_qsvm()
        CLFS['QSVM (ZZ)'].fit(X_train, y_train)
        print(f'       Done in {time.time()-t0:.1f}s')
        for name, kwargs, Xtr in [
            ('SVM-RBF (mm)',     dict(kernel='rbf',    C=SVM_C_BY_MODEL['SVM-RBF (mm)']),       X_train),
            ('SVM-Poly2 (mm)',   dict(kernel='poly', degree=2, C=SVM_C_BY_MODEL['SVM-Poly2 (mm)']), X_train),
            ('SVM-Linear (mm)',  dict(kernel='linear', C=SVM_C_BY_MODEL['SVM-Linear (mm)']),   X_train),
            ('SVM-RBF (std)',    dict(kernel='rbf',    C=SVM_C_BY_MODEL['SVM-RBF (std)']),      X_train_std),
            ('SVM-Poly2 (std)',  dict(kernel='poly', degree=2, C=SVM_C_BY_MODEL['SVM-Poly2 (std)']), X_train_std),
            ('SVM-Linear (std)', dict(kernel='linear', C=SVM_C_BY_MODEL['SVM-Linear (std)']),  X_train_std),
        ]:
            clf = SVC(probability=True, random_state=RANDOM_STATE, **kwargs)
            clf.fit(Xtr, y_train)
            CLFS[name] = clf
        joblib.dump({'classifiers': CLFS, 'std_scaler': std_scaler}, model_path)
        print(f'[C4][run_{run_id}] models trained & cached')

    temporal_rows, perturb_rows, slope_rows, prior_rows, mcnemar_rows = [], [], [], [], []

    # ── E1: Temporal split ────────────────────────────────────────────────────
    X_e1_std    = transform_pipeline(df_e1_std, feature_cols)
    X_e1_std_sc = std_scaler.transform(pca_only(df_e1_std, feature_cols))
    X_e1_21     = transform_pipeline(df_e1_21, feature_cols)
    X_e1_21_sc  = std_scaler.transform(pca_only(df_e1_21, feature_cols))
    y_e1_std    = df_e1_std['label_binary'].to_numpy(dtype=np.int64)
    y_e1_21     = df_e1_21['label_binary'].to_numpy(dtype=np.int64)

    hard_preds = {}
    for name, clf in CLFS.items():
        Xs = X_e1_std_sc if '(std)' in name else X_e1_std
        Xh = X_e1_21_sc  if '(std)' in name else X_e1_21
        ps, ph = clf.predict(Xs), clf.predict(Xh)
        f1s = f1_score(y_e1_std, ps, average='macro')
        f1h = f1_score(y_e1_21,  ph, average='macro')
        drop = f1s - f1h
        temporal_rows.append({'run_id': run_id, 'model_name': name,
                              'f1_standard': f1s, 'f1_hard': f1h,
                              'drop_abs': drop,
                              'drop_pct': drop / max(f1s, 1e-9) * 100})
        hard_preds[name] = ph

    # McNemar: QSVM vs all baselines on hard test
    for name in [n for n in CLFS if n != 'QSVM (ZZ)']:
        chi2, p, nd = mcnemar_test(y_e1_21, hard_preds['QSVM (ZZ)'], hard_preds[name])
        mcnemar_rows.append({'run_id': run_id, 'baseline': name,
                             'chi2': chi2, 'p_value': p, 'n_discordant': nd})
    print(f'[C4][run_{run_id}] E1 done')

    # ── E2: Feature perturbation ──────────────────────────────────────────────
    X_e2     = transform_pipeline(df_e2_base, feature_cols)
    X_e2_pca = pca_only(df_e2_base, feature_cols)
    y_e2     = df_e2_base['label_binary'].to_numpy(dtype=np.int64)
    rng2     = np.random.RandomState(RANDOM_STATE + 100)

    e2_by_model = {n: [] for n in CLFS}
    for sigma in SIGMA_LEVELS:
        if sigma == 0.0:
            Xmm  = X_e2.copy()
            Xstd = std_scaler.transform(X_e2_pca)
        else:
            noise = rng2.normal(0, sigma, size=X_e2_pca.shape)
            Xp    = X_e2_pca + noise
            Xmm   = np.clip(scaler.transform(Xp), 0, ANGLE_MAX)
            Xstd  = std_scaler.transform(Xp)
        for name, clf in CLFS.items():
            Xi  = Xstd if '(std)' in name else Xmm
            f1  = f1_score(y_e2, clf.predict(Xi), average='macro')
            perturb_rows.append({'run_id': run_id, 'model_name': name,
                                 'sigma': sigma, 'f1_macro': f1})
            e2_by_model[name].append(f1)

    for name, vals in e2_by_model.items():
        slope_rows.append({'run_id': run_id, 'model_name': name,
                           'slope': np.polyfit(SIGMA_LEVELS, vals, 1)[0]})
    print(f'[C4][run_{run_id}] E2 done')

    # ── E3: Class prior shift ─────────────────────────────────────────────────
    for dist_name, df_e3 in test_sets_e3.items():
        Xmm  = transform_pipeline(df_e3, feature_cols)
        Xstd = std_scaler.transform(pca_only(df_e3, feature_cols))
        y    = df_e3['label_binary'].to_numpy(dtype=np.int64)
        for name, clf in CLFS.items():
            pred = clf.predict(Xstd if '(std)' in name else Xmm)
            prior_rows.append({'run_id': run_id, 'model_name': name,
                               'distribution_name': dist_name,
                               'f1_macro': f1_score(y, pred, average='macro'),
                               'accuracy': accuracy_score(y, pred)})
    print(f'[C4][run_{run_id}] E3 done')
    print(f'[C4][run_{run_id}] ✓ complete')
    return temporal_rows, perturb_rows, slope_rows, prior_rows, mcnemar_rows

## 5. Run 5 Independent Training Runs & Aggregate Metrics

Lặp qua 5 training splits, gom kết quả per-run, sau đó tính `mean ± std` cho từng model và từng experiment.


In [ ]:
# ─── Run 5 independent training runs ─────────────────────────────────────────
temporal_all, perturb_all, slope_all, prior_all, mcnemar_all = [], [], [], [], []

for run_id in RUN_IDS:
    train_path = f'{DATA_DIR}/multi_run/train_run{run_id}.csv'
    assert os.path.exists(train_path), f'Missing: {train_path}'
    t, p, s, c, m = run_c4_single_train(train_path, run_id)
    temporal_all.extend(t); perturb_all.extend(p); slope_all.extend(s)
    prior_all.extend(c);   mcnemar_all.extend(m)

temporal_df = pd.DataFrame(temporal_all)
perturb_df  = pd.DataFrame(perturb_all)
slope_df    = pd.DataFrame(slope_all)
prior_df    = pd.DataFrame(prior_all)
mcnemar_df  = pd.DataFrame(mcnemar_all)

# ─── Aggregate: mean ± std per model ─────────────────────────────────────────
temporal_summary = (
    temporal_df.groupby('model_name')
    .agg(f1_standard_mean=('f1_standard','mean'), f1_standard_std=('f1_standard','std'),
         f1_hard_mean=('f1_hard','mean'),         f1_hard_std=('f1_hard','std'),
         drop_abs_mean=('drop_abs','mean'),        drop_abs_std=('drop_abs','std'),
         drop_pct_mean=('drop_pct','mean'),        drop_pct_std=('drop_pct','std'))
    .reindex(MODEL_ORDER).reset_index()
)
perturbation_summary = (
    perturb_df.groupby(['model_name','sigma'])
    .agg(f1_macro_mean=('f1_macro','mean'), f1_macro_std=('f1_macro','std'))
    .reset_index()
)
slope_summary = (
    slope_df.groupby('model_name')
    .agg(slope_mean=('slope','mean'), slope_std=('slope','std'))
    .reindex(MODEL_ORDER).reset_index()
)
prior_summary = (
    prior_df.groupby(['model_name','distribution_name'])
    .agg(f1_macro_mean=('f1_macro','mean'), f1_macro_std=('f1_macro','std'))
    .reset_index()
)
mcnemar_summary = (
    mcnemar_df.groupby('baseline')
    .agg(chi2_mean=('chi2','mean'), chi2_std=('chi2','std'),
         p_value_mean=('p_value','mean'), p_value_std=('p_value','std'))
    .reset_index()
)

# ─── Save CSVs ────────────────────────────────────────────────────────────────
temporal_summary.to_csv(f'{OUTPUT_DIR}/temporal_shift_summary.csv',    index=False)
perturbation_summary.to_csv(f'{OUTPUT_DIR}/perturbation_summary.csv',  index=False)
slope_summary.to_csv(f'{OUTPUT_DIR}/degradation_slope_summary.csv',    index=False)
prior_summary.to_csv(f'{OUTPUT_DIR}/class_prior_shift_summary.csv',    index=False)
mcnemar_summary.to_csv(f'{OUTPUT_DIR}/mcnemar_summary.csv',            index=False)

print('\n── E1 Temporal Summary (mean ± std over 5 runs) ────────────────────')
print(temporal_summary[['model_name','f1_standard_mean','f1_standard_std',
                         'f1_hard_mean','f1_hard_std','drop_abs_mean','drop_abs_std']].to_string(index=False))
print('\n── E2 Slope Summary ────────────────────────────────────────────────')
print(slope_summary.to_string(index=False))


[C4][run_1] start → ../data/processed_data/multi_run/train_run1.csv
  1/7  QSVM (ZZFeatureMap) ...


## 6. Pretty Summary Tables — Mean ± Std

Hiển thị các bảng tóm tắt rõ ràng hơn cho E1, E2, E3 và McNemar test.


In [ ]:
# ─── Pretty tables: mean ± std summaries ────────────────────────────────────
def fmt_mean_std(mean_val, std_val, digits=4):
    if pd.isna(mean_val):
        return 'NA'
    if pd.isna(std_val):
        return f'{mean_val:.{digits}f}'
    return f'{mean_val:.{digits}f} ± {std_val:.{digits}f}'

def display_pretty_table(df, caption, subset=None):
    styled = (
        df.style
        .hide(axis='index')
        .set_caption(caption)
        .set_table_styles([
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'left'), ('padding', '8px 0')]},
            {'selector': 'th', 'props': [('background-color', '#263238'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '7px 10px')]},
            {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '6px 10px'), ('border-bottom', '1px solid #ECEFF1')]},
        ])
    )
    if subset:
        styled = styled.background_gradient(cmap='YlGnBu', subset=subset)
    display(styled)

# E1: Temporal shift summary
temporal_pretty = temporal_summary.copy()
temporal_pretty['Standard F1'] = temporal_pretty.apply(lambda r: fmt_mean_std(r.f1_standard_mean, r.f1_standard_std), axis=1)
temporal_pretty['Hard F1'] = temporal_pretty.apply(lambda r: fmt_mean_std(r.f1_hard_mean, r.f1_hard_std), axis=1)
temporal_pretty['Drop abs'] = temporal_pretty.apply(lambda r: fmt_mean_std(r.drop_abs_mean, r.drop_abs_std), axis=1)
temporal_pretty['Drop %'] = temporal_pretty.apply(lambda r: fmt_mean_std(r.drop_pct_mean, r.drop_pct_std), axis=1)
temporal_pretty = temporal_pretty[['model_name', 'Standard F1', 'Hard F1', 'Drop abs', 'Drop %']]
temporal_pretty = temporal_pretty.rename(columns={'model_name': 'Model'})

display_pretty_table(
    temporal_pretty,
    'E1 Temporal Shift — mean ± std over 5 runs'
)

# E2: Perturbation robustness by sigma
perturb_pretty = perturbation_summary.copy()
perturb_pretty['sigma'] = perturb_pretty['sigma'].map(lambda v: f'{v:g}')
perturb_pretty['F1 macro'] = perturb_pretty.apply(lambda r: fmt_mean_std(r.f1_macro_mean, r.f1_macro_std), axis=1)
perturb_pretty = (
    perturb_pretty[['model_name', 'sigma', 'F1 macro']]
    .pivot(index='model_name', columns='sigma', values='F1 macro')
    .reindex(MODEL_ORDER)
    .reset_index()
    .rename(columns={'model_name': 'Model'})
)

display_pretty_table(
    perturb_pretty,
    'E2 Feature Perturbation — F1 macro mean ± std by sigma'
)

# E2 degradation slope summary
slope_pretty = slope_summary.copy()
slope_pretty['Degradation slope'] = slope_pretty.apply(lambda r: fmt_mean_std(r.slope_mean, r.slope_std), axis=1)
slope_pretty = slope_pretty[['model_name', 'Degradation slope']].rename(columns={'model_name': 'Model'})

display_pretty_table(
    slope_pretty,
    'E2 Degradation Slope — mean ± std over 5 runs'
)

# E3: Class prior shift summary
prior_pretty = prior_summary.copy()
prior_pretty['F1 macro'] = prior_pretty.apply(lambda r: fmt_mean_std(r.f1_macro_mean, r.f1_macro_std), axis=1)
prior_pretty = (
    prior_pretty[['model_name', 'distribution_name', 'F1 macro']]
    .pivot(index='model_name', columns='distribution_name', values='F1 macro')
    .reindex(MODEL_ORDER)
    .reset_index()
    .rename(columns={'model_name': 'Model'})
)

display_pretty_table(
    prior_pretty,
    'E3 Class Prior Shift — F1 macro mean ± std by distribution'
)

# McNemar summary
mcnemar_pretty = mcnemar_summary.copy()
mcnemar_pretty['chi2'] = mcnemar_pretty.apply(lambda r: fmt_mean_std(r.chi2_mean, r.chi2_std), axis=1)
mcnemar_pretty['p-value'] = mcnemar_pretty.apply(lambda r: fmt_mean_std(r.p_value_mean, r.p_value_std), axis=1)
mcnemar_pretty = mcnemar_pretty[['baseline', 'chi2', 'p-value']].rename(columns={'baseline': 'Baseline vs QSVM'})

display_pretty_table(
    mcnemar_pretty,
    'McNemar Test — mean ± std over 5 runs'
)


## 7. Experiment 1 — Temporal Split Evaluation

**Protocol:** train trên từng `train_run{i}`, evaluate cùng lúc trên **KDDTest+** và **KDDTest-21**. Mục tiêu là đo mức suy giảm F1 khi chuyển sang hard temporal split.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Figure E1 — Temporal Split: F1 comparison + F1 drop  (optimized)
# ═══════════════════════════════════════════════════════════════════════════════
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.dpi': 120,
})

def _clean_axis(ax, grid_axis='y'):
    ax.grid(True, axis=grid_axis, alpha=0.22, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines['left'].set_color('#CFD8DC')
    ax.spines['bottom'].set_color('#CFD8DC')

def _label_bars(ax, bars, values, errors=None, fmt='{:.3f}', fontsize=8, dy=0.006):
    if errors is None:
        errors = np.zeros(len(values))
    for bar, val, err in zip(bars, values, errors):
        if pd.isna(val):
            continue
        y = val + (err if not pd.isna(err) else 0) + dy
        va = 'bottom'
        if val < 0:
            y = val - (err if not pd.isna(err) else 0) - dy
            va = 'top'
        ax.text(bar.get_x() + bar.get_width()/2, y, fmt.format(val),
                ha='center', va=va, fontsize=fontsize, color='#263238', fontweight='bold')

tmp = temporal_summary.set_index('model_name').reindex(MODEL_ORDER).reset_index()
x = np.arange(len(tmp))
colors = [PLOT_COLORS[m] for m in tmp['model_name']]

fig, axes = plt.subplots(1, 2, figsize=(18, 6.2), gridspec_kw={'width_ratios': [1.25, 1]})
fig.suptitle(
    'E1 — Temporal Split Robustness\n'
    f'KDDTest+ vs KDDTest-21, mean ± std over {len(RUN_IDS)} independent training runs',
    fontsize=14, fontweight='bold', y=1.03
)

# ── Left: grouped F1 bar chart ───────────────────────────────────────────────
ax = axes[0]
y = np.arange(len(tmp))
std_mu = tmp['f1_standard_mean'].values
hard_mu = tmp['f1_hard_mean'].values
std_sd = tmp['f1_standard_std'].fillna(0).values
hard_sd = tmp['f1_hard_std'].fillna(0).values
width = 0.36

b_std = ax.bar(
    y - width/2,
    std_mu,
    width,
    yerr=std_sd,
    capsize=4,
    color=colors,
    alpha=0.45,
    edgecolor='white',
    linewidth=1.2,
    label='KDDTest+ (Standard)',
    error_kw=dict(elinewidth=1.2, ecolor='#455A64', capthick=1.1),
    zorder=3,
)
b_hard = ax.bar(
    y + width/2,
    hard_mu,
    width,
    yerr=hard_sd,
    capsize=4,
    color=colors,
    alpha=0.92,
    edgecolor='white',
    linewidth=1.2,
    label='KDDTest-21 (Hard)',
    error_kw=dict(elinewidth=1.2, ecolor='#455A64', capthick=1.1),
    zorder=3,
)

_label_bars(ax, b_std, std_mu, std_sd, fmt='{:.3f}', fontsize=7.8, dy=0.009)
_label_bars(ax, b_hard, hard_mu, hard_sd, fmt='{:.3f}', fontsize=8.0, dy=0.009)

for i, row in tmp.iterrows():
    y_top = max(row['f1_standard_mean'] + row['f1_standard_std'], row['f1_hard_mean'] + row['f1_hard_std']) + 0.055
    ax.plot([i - width/2, i + width/2], [y_top, y_top], color='#78909C', linewidth=1.0)
    ax.text(i, y_top + 0.010, f'Δ={row["drop_abs_mean"]:+.3f}',
            ha='center', va='bottom', fontsize=7.8, color='#455A64')

ax.set_xticks(y)
ax.set_xticklabels(tmp['model_name'], rotation=22, ha='right', fontsize=9)
ax.set_ylabel('F1-macro')
ax.set_ylim(max(0.0, np.nanmin([hard_mu - hard_sd, std_mu - std_sd]) - 0.06), 1.08)
ax.set_title('F1-macro: KDDTest+ vs KDDTest-21')
ax.legend(
    handles=[
        mpatches.Patch(facecolor='#78909C', alpha=0.45, edgecolor='white', label='KDDTest+ (Standard)'),
        mpatches.Patch(facecolor='#78909C', alpha=0.92, edgecolor='white', label='KDDTest-21 (Hard)'),
    ],
    loc='lower left', frameon=True, fontsize=9,
)
_clean_axis(ax)

# ── Right: absolute F1 drop ──────────────────────────────────────────────────
ax2 = axes[1]
drops = tmp['drop_abs_mean'].values
errs = tmp['drop_abs_std'].fillna(0).values
bars = ax2.barh(y, drops, xerr=errs, color=colors, alpha=0.88,
                edgecolor='white', linewidth=1.2, capsize=4,
                error_kw=dict(elinewidth=1.3, ecolor='#455A64', capthick=1.2), zorder=3)
ax2.axvline(0, color='#263238', linewidth=1.0)
for i, (v, e) in enumerate(zip(drops, errs)):
    x_text = v + e + 0.006 if v >= 0 else v - e - 0.006
    ha = 'left' if v >= 0 else 'right'
    ax2.text(x_text, i, f'{v:+.3f}', ha=ha, va='center', fontsize=8.5, fontweight='bold', color='#263238')
ax2.set_yticks(y)
ax2.set_yticklabels([])
ax2.invert_yaxis()
ax2.set_xlabel('F1 drop = F1_standard − F1_hard')
ax2.set_title('Temporal Drop (lower is better)')
_clean_axis(ax2, grid_axis='x')

plt.tight_layout()
path = f'{FIG_DIR}/c4_e1_temporal_split_multirun.png'
plt.savefig(path, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {path}')

# ── Figure E1b: per-run drop strip + mean bar ────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(14, 5.4))
bar_x = np.arange(len(tmp))
bars = ax3.bar(bar_x, drops, color=colors, alpha=0.26, edgecolor='white', linewidth=1.1, zorder=2)
ax3.errorbar(bar_x, drops, yerr=errs, fmt='none', ecolor='#455A64', elinewidth=1.25, capsize=4, zorder=3)
for run_id in RUN_IDS:
    sub = temporal_df[temporal_df['run_id'] == run_id].set_index('model_name').reindex(MODEL_ORDER)
    jitter = (run_id - np.mean(RUN_IDS)) * 0.035
    ax3.scatter(bar_x + jitter, sub['drop_abs'], s=42, color=colors,
                edgecolor='white', linewidth=0.8, alpha=0.86, zorder=4,
                label=f'run {run_id}' if run_id == RUN_IDS[0] else None)
_label_bars(ax3, bars, drops, errs, fmt='{:+.3f}', fontsize=8)
ax3.axhline(0, color='#263238', linewidth=1.0)
ax3.set_xticks(bar_x)
ax3.set_xticklabels(MODEL_ORDER, rotation=22, ha='right', fontsize=9)
ax3.set_ylabel('F1 drop per run')
ax3.set_title('E1b — Temporal Drop Distribution Across Runs')
_clean_axis(ax3)
plt.tight_layout()
path3 = f'{FIG_DIR}/c4_e1_temporal_drop_per_run.png'
plt.savefig(path3, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {path3}')


## 8. Experiment 2 — Feature Perturbation Robustness

**Protocol:** thêm nhiễu Gaussian vào feature space sau pipeline C1 với nhiều mức `sigma`, rồi đo đường suy giảm F1-macro và degradation slope.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Figure E2 — Feature Perturbation Robustness  (optimized)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 6.2), gridspec_kw={'width_ratios': [1.25, 1]})
fig.suptitle(
    'E2 — Feature Perturbation Robustness\n'
    f'F1-macro vs Gaussian noise σ, mean ± std over {len(RUN_IDS)} runs',
    fontsize=14, fontweight='bold', y=1.03
)

# ── Left: absolute F1 curves with shaded ±std band ───────────────────────────
ax = axes[0]
for clf_name in MODEL_ORDER:
    d = perturbation_summary[perturbation_summary['model_name'] == clf_name].sort_values('sigma')
    if d.empty:
        continue
    mu = d['f1_macro_mean'].values
    sd = d['f1_macro_std'].fillna(0).values
    sigs = d['sigma'].values
    is_q = clf_name == 'QSVM (ZZ)'
    ax.plot(sigs, mu,
            color=PLOT_COLORS[clf_name], marker=MARKERS[clf_name], linestyle=LINESTYLES[clf_name],
            linewidth=3.0 if is_q else 1.8, markersize=7 if is_q else 5.5,
            markeredgecolor='white', markeredgewidth=0.9, label=clf_name, zorder=4 if is_q else 3)
    ax.fill_between(sigs, mu - sd, mu + sd,
                    alpha=0.18 if is_q else 0.09, color=PLOT_COLORS[clf_name], linewidth=0)
    ax.text(sigs[-1] + 0.006, mu[-1], f'{mu[-1]:.3f}',
            va='center', fontsize=8, color=PLOT_COLORS[clf_name], fontweight='bold' if is_q else 'normal')

ax.axvspan(0, 0.05, alpha=0.07, color='#2E7D32', linewidth=0)
ax.axvline(0.05, color='#2E7D32', linestyle=':', alpha=0.75, linewidth=1.2)
ax.text(0.025, ax.get_ylim()[0] + 0.02, 'low-noise\nregion', ha='center', va='bottom', fontsize=8, color='#2E7D32')
ax.set_xlabel('Gaussian noise σ')
ax.set_ylabel('F1-macro')
ax.set_title('Absolute Robustness Curve')
ax.set_xlim(-0.005, max(SIGMA_LEVELS) + 0.035)
ax.set_ylim(max(0.0, perturbation_summary['f1_macro_mean'].min() - 0.10), 1.03)
_clean_axis(ax)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8.5, frameon=True, borderaxespad=0)

# ── Right: relative F1 drop by sigma ─────────────────────────────────────────
ax2 = axes[1]
rel_rows = []
for (run_id, clf_name), g in perturb_df.groupby(['run_id', 'model_name']):
    g = g.sort_values('sigma')
    base_vals = g[g['sigma'] == 0.0]['f1_macro'].values
    if len(base_vals) == 0 or abs(base_vals[0]) < 1e-12:
        continue
    base = base_vals[0]
    for _, row in g.iterrows():
        rel_rows.append({
            'run_id': run_id,
            'model_name': clf_name,
            'sigma': row['sigma'],
            'drop_pct': 100.0 * (base - row['f1_macro']) / base,
        })
rel_drop_summary = (
    pd.DataFrame(rel_rows)
    .groupby(['model_name', 'sigma'])
    .agg(drop_pct_mean=('drop_pct', 'mean'), drop_pct_std=('drop_pct', 'std'))
    .reset_index()
)

for clf_name in MODEL_ORDER:
    d = rel_drop_summary[rel_drop_summary['model_name'] == clf_name].sort_values('sigma')
    if d.empty:
        continue
    sigs = d['sigma'].values
    mu = d['drop_pct_mean'].values
    sd = d['drop_pct_std'].fillna(0).values
    is_q = clf_name == 'QSVM (ZZ)'
    ax2.plot(sigs, mu, color=PLOT_COLORS[clf_name], marker=MARKERS[clf_name], linestyle=LINESTYLES[clf_name],
             linewidth=2.8 if is_q else 1.7, markersize=6 if is_q else 5,
             markeredgecolor='white', markeredgewidth=0.8, label=clf_name, zorder=4 if is_q else 3)
    ax2.fill_between(sigs, mu - sd, mu + sd, color=PLOT_COLORS[clf_name], alpha=0.14 if is_q else 0.07, linewidth=0)

ax2.axhline(0, color='#263238', linewidth=1.0)
ax2.axvline(0.05, color='#2E7D32', linestyle=':', alpha=0.75, linewidth=1.2)
ax2.set_xlabel('Gaussian noise σ')
ax2.set_ylabel('Relative F1 drop (%)')
ax2.set_title('Relative Degradation (lower is better)')
ax2.set_xlim(-0.005, max(SIGMA_LEVELS) + 0.01)
_clean_axis(ax2)

plt.tight_layout()
path = f'{FIG_DIR}/c4_e2_perturbation_robustness_multirun.png'
plt.savefig(path, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {path}')

# ── Figure E2b: degradation slope ranking ───────────────────────────────────
sl = slope_summary.set_index('model_name').reindex(MODEL_ORDER).reset_index()
sl = sl.sort_values('slope_mean', ascending=True)
fig2, ax3 = plt.subplots(figsize=(12.5, 5.8))
y = np.arange(len(sl))
colors = [PLOT_COLORS[m] for m in sl['model_name']]
ax3.barh(y, sl['slope_mean'], xerr=sl['slope_std'].fillna(0), color=colors, alpha=0.88,
         edgecolor='white', linewidth=1.2, capsize=4,
         error_kw=dict(elinewidth=1.25, ecolor='#455A64', capthick=1.1), zorder=3)
ax3.axvline(0, color='#263238', linewidth=1.0)
for i, row in sl.iterrows():
    e = 0 if pd.isna(row['slope_std']) else row['slope_std']
    x_text = row['slope_mean'] - e - 0.015 if row['slope_mean'] < 0 else row['slope_mean'] + e + 0.015
    ha = 'right' if row['slope_mean'] < 0 else 'left'
    ax3.text(x_text, i, f'{row["slope_mean"]:+.3f}', va='center', ha=ha, fontsize=8.5, fontweight='bold')
ax3.set_yticks(y)
ax3.set_yticklabels(sl['model_name'], fontsize=9.5)
ax3.set_xlabel('Slope of F1 vs σ')
ax3.set_title('E2b — Degradation Slope Ranking (closer to 0 is more robust)')
_clean_axis(ax3, grid_axis='x')
plt.tight_layout()
path2 = f'{FIG_DIR}/c4_e2_degradation_slope_multirun.png'
plt.savefig(path2, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {path2}')


## 9. Experiment 3 — Class Prior Shift

**Protocol:** đánh giá các model trên nhiều phân phối class prior khác nhau để kiểm tra độ ổn định khi tỷ lệ Normal/Attack thay đổi.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Figure E3 — Class Prior Shift  (optimized)
# ═══════════════════════════════════════════════════════════════════════════════
set_names = E3_SET_NAMES_DISPLAY
set_names_clean = [s.replace('\n', ' ') for s in set_names]

# Build F1-macro matrix (rows=classifiers, cols=distributions)
f1_matrix_mu = np.array([
    [prior_summary[(prior_summary['model_name'] == clf) &
                   (prior_summary['distribution_name'] == sn)]['f1_macro_mean'].values[0]
     for sn in set_names]
    for clf in MODEL_ORDER
])
f1_matrix_sd = np.array([
    [prior_summary[(prior_summary['model_name'] == clf) &
                   (prior_summary['distribution_name'] == sn)]['f1_macro_std'].values[0]
     for sn in set_names]
    for clf in MODEL_ORDER
])

e3_mean_f1 = np.nanmean(f1_matrix_mu, axis=1)
e3_std_f1 = np.nanstd(f1_matrix_mu, axis=1, ddof=1)
e3_avg_run_std = np.nanmean(f1_matrix_sd, axis=1)

annot = np.empty(f1_matrix_mu.shape, dtype=object)
for i in range(f1_matrix_mu.shape[0]):
    for j in range(f1_matrix_mu.shape[1]):
        annot[i, j] = f'{f1_matrix_mu[i, j]:.3f}\n±{f1_matrix_sd[i, j]:.3f}'

fig, axes = plt.subplots(1, 2, figsize=(19, 6.6), gridspec_kw={'width_ratios': [1.15, 1]})
fig.suptitle(
    'E3 — Class Prior Shift Robustness\n'
    f'F1-macro across test distributions, mean ± std over {len(RUN_IDS)} runs',
    fontsize=14, fontweight='bold', y=1.03
)

# ── Left: heatmap ────────────────────────────────────────────────────────────
ax = axes[0]
sns.heatmap(
    f1_matrix_mu,
    annot=annot,
    fmt='',
    cmap='RdYlGn',
    vmin=max(0.0, np.nanmin(f1_matrix_mu) - 0.05),
    vmax=min(1.0, np.nanmax(f1_matrix_mu) + 0.04),
    linewidths=0.8,
    linecolor='white',
    xticklabels=set_names_clean,
    yticklabels=MODEL_ORDER,
    cbar_kws={'label': 'F1-macro mean', 'shrink': 0.86},
    annot_kws={'fontsize': 8.5, 'fontweight': 'bold'},
    ax=ax,
)
ax.set_title('F1-macro by Distribution')
ax.set_xlabel('Test distribution')
ax.set_ylabel('Model')
ax.tick_params(axis='x', rotation=18)
ax.tick_params(axis='y', rotation=0)
qrow = MODEL_ORDER.index('QSVM (ZZ)')
ax.add_patch(plt.Rectangle((0, qrow), len(set_names), 1, fill=False, edgecolor='#1565C0', linewidth=2.2, clip_on=False))

# ── Right: model mean across distributions + distribution spread ─────────────
ax2 = axes[1]
order_idx = np.argsort(e3_mean_f1)
y = np.arange(len(MODEL_ORDER))
ordered_models = [MODEL_ORDER[i] for i in order_idx]
ordered_mean = e3_mean_f1[order_idx]
ordered_spread = e3_std_f1[order_idx]
ordered_run_std = e3_avg_run_std[order_idx]
ordered_colors = [PLOT_COLORS[m] for m in ordered_models]

ax2.barh(y, ordered_mean, xerr=ordered_run_std, color=ordered_colors, alpha=0.88,
         edgecolor='white', linewidth=1.2, capsize=4,
         error_kw=dict(elinewidth=1.25, ecolor='#455A64', capthick=1.1), zorder=3)
for i, (mu, spread) in enumerate(zip(ordered_mean, ordered_spread)):
    ax2.text(mu + ordered_run_std[i] + 0.008, i, f'{mu:.3f}\nspread={spread:.3f}',
             va='center', ha='left', fontsize=8.2, color='#263238')
ax2.set_yticks(y)
ax2.set_yticklabels(ordered_models, fontsize=9.5)
ax2.set_xlabel('Mean F1 across distributions')
ax2.set_title('Average Robustness + Prior Sensitivity')
ax2.set_xlim(max(0, np.nanmin(ordered_mean - ordered_run_std) - 0.06), min(1.04, np.nanmax(ordered_mean + ordered_run_std) + 0.13))
_clean_axis(ax2, grid_axis='x')

plt.tight_layout()
path = f'{FIG_DIR}/c4_e3_prior_shift_multirun.png'
plt.savefig(path, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {path}')

# ── Figure E3b: stability score ──────────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(13.5, 5.6))
x_pos = np.arange(len(MODEL_ORDER))
colors = [PLOT_COLORS[m] for m in MODEL_ORDER]
bars = ax3.bar(x_pos, e3_mean_f1, color=colors, alpha=0.86, edgecolor='white', linewidth=1.2, zorder=3)
ax3.errorbar(x_pos, e3_mean_f1, yerr=e3_std_f1, fmt='none', ecolor='#263238', elinewidth=1.35, capsize=5, zorder=4,
             label='std across prior distributions')
for i, (mu, spread) in enumerate(zip(e3_mean_f1, e3_std_f1)):
    ax3.text(i, mu + spread + 0.008, f'{mu:.3f}\n±{spread:.3f}', ha='center', va='bottom', fontsize=8.2, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(MODEL_ORDER, rotation=22, ha='right', fontsize=9)
ax3.set_ylabel('F1-macro')
ax3.set_title('E3b — Stability Across Class Prior Shifts')
ax3.set_ylim(max(0, np.nanmin(e3_mean_f1 - e3_std_f1) - 0.08), min(1.05, np.nanmax(e3_mean_f1 + e3_std_f1) + 0.12))
_clean_axis(ax3)
ax3.legend(fontsize=9, loc='lower right')
qidx3 = MODEL_ORDER.index('QSVM (ZZ)')
ax3.patches[qidx3].set_linewidth(2.4)
ax3.patches[qidx3].set_edgecolor('#1565C0')
plt.tight_layout()
path3 = f'{FIG_DIR}/c4_e3_stability_multirun.png'
plt.savefig(path3, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {path3}')


## 10. Tổng Hợp Kết Quả C4 — Robustness Summary

Tổng hợp ba góc nhìn robustness chính vào một hình: temporal drop, perturbation curve và prior-shift average.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Master Summary Figure — 3 panels (E1 / E2 / E3)  (optimized)
# ═══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(21, 6.8))
fig.suptitle(
    'C4 — Robustness Summary: Distribution Shift Evaluation\n'
    f'QSVM vs Classical SVM, mean ± std over {len(RUN_IDS)} independent training runs',
    fontsize=14, fontweight='bold', y=1.03
)
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.36, width_ratios=[1, 1.25, 1])

# ── Panel 1: E1 temporal F1-drop ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
tmp = temporal_summary.set_index('model_name').reindex(MODEL_ORDER).reset_index()
drops = tmp['drop_abs_mean'].values
errs = tmp['drop_abs_std'].fillna(0).values
bar_c = [PLOT_COLORS[m] for m in MODEL_ORDER]
y = np.arange(len(MODEL_ORDER))
ax1.barh(y, drops, xerr=errs, color=bar_c, alpha=0.88, edgecolor='white', linewidth=1.1, capsize=4,
         error_kw=dict(elinewidth=1.2, ecolor='#455A64', capthick=1.1), zorder=3)
ax1.axvline(0, color='#263238', linewidth=1.0)
for i, (v, e) in enumerate(zip(drops, errs)):
    ax1.text(v + e + 0.006, i, f'{v:+.3f}', ha='left', va='center', fontsize=8, fontweight='bold')
ax1.set_yticks(y)
ax1.set_yticklabels(MODEL_ORDER, fontsize=8.8)
ax1.invert_yaxis()
ax1.set_xlabel('ΔF1')
ax1.set_title('E1 Temporal Drop\nlower is better', fontsize=10.5)
_clean_axis(ax1, grid_axis='x')

# ── Panel 2: E2 degradation curves ───────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
for clf_name in MODEL_ORDER:
    d = perturbation_summary[perturbation_summary['model_name'] == clf_name].sort_values('sigma')
    if d.empty:
        continue
    mu = d['f1_macro_mean'].values
    sd = d['f1_macro_std'].fillna(0).values
    sigs = d['sigma'].values
    is_q = clf_name == 'QSVM (ZZ)'
    ax2.plot(sigs, mu, color=PLOT_COLORS[clf_name], linestyle=LINESTYLES[clf_name], linewidth=2.8 if is_q else 1.5,
             marker=MARKERS[clf_name], markersize=5.8 if is_q else 4.6, markeredgecolor='white', markeredgewidth=0.7,
             label=clf_name, zorder=4 if is_q else 3)
    ax2.fill_between(sigs, mu - sd, mu + sd, alpha=0.16 if is_q else 0.06, color=PLOT_COLORS[clf_name], linewidth=0)
ax2.axvline(0.05, color='#2E7D32', linestyle=':', alpha=0.75, linewidth=1.2)
ax2.set_xlabel('Noise σ')
ax2.set_ylabel('F1-macro')
ax2.set_title('E2 Perturbation Robustness\nhigher curve is better', fontsize=10.5)
ax2.set_xlim(-0.005, max(SIGMA_LEVELS) + 0.01)
ax2.set_ylim(max(0, perturbation_summary['f1_macro_mean'].min() - 0.08), 1.03)
_clean_axis(ax2)
ax2.legend(fontsize=7.5, loc='lower left', ncol=1, frameon=True)

# ── Panel 3: E3 average F1 across prior shifts ───────────────────────────────
ax3 = fig.add_subplot(gs[2])
if 'e3_mean_f1' not in globals():
    set_names = E3_SET_NAMES_DISPLAY
    f1_matrix_mu = np.array([
        [prior_summary[(prior_summary['model_name'] == clf) &
                       (prior_summary['distribution_name'] == sn)]['f1_macro_mean'].values[0]
         for sn in set_names]
        for clf in MODEL_ORDER
    ])
    e3_mean_f1 = np.nanmean(f1_matrix_mu, axis=1)
    e3_std_f1 = np.nanstd(f1_matrix_mu, axis=1, ddof=1)

ax3.barh(y, e3_mean_f1, xerr=e3_std_f1, color=bar_c, alpha=0.88, edgecolor='white', linewidth=1.1, capsize=4,
         error_kw=dict(elinewidth=1.2, ecolor='#455A64', capthick=1.1), zorder=3)
for i, (v, e) in enumerate(zip(e3_mean_f1, e3_std_f1)):
    ax3.text(v + e + 0.008, i, f'{v:.3f}', ha='left', va='center', fontsize=8, fontweight='bold')
ax3.set_yticks(y)
ax3.set_yticklabels([])
ax3.invert_yaxis()
ax3.set_xlabel('Mean F1')
ax3.set_title('E3 Prior-Shift Average\nhigher is better', fontsize=10.5)
ax3.set_xlim(max(0, np.nanmin(e3_mean_f1 - e3_std_f1) - 0.05), min(1.05, np.nanmax(e3_mean_f1 + e3_std_f1) + 0.12))
_clean_axis(ax3, grid_axis='x')

plt.tight_layout()
summary_path = f'{FIG_DIR}/c4_robustness_summary_multirun.png'
plt.savefig(summary_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'[SAVED] {summary_path}')


## 11. Quantitative Results Table

Bảng số liệu cuối cùng cho báo cáo: E1 temporal split, E2 degradation slope, E3 prior shift và McNemar significance.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Quantitative Results Table — mean ± std
# ═══════════════════════════════════════════════════════════════════════════════
print('=' * 90)
print(f'C4 MULTI-RUN QUANTITATIVE RESULTS TABLE  ({len(RUN_IDS)} runs, fixed test sets)')
print('=' * 90)

print('\n── E1: Temporal Split ──────────────────────────────────────────────────────────────')
hdr = f'{"Classifier":18s}  {"F1_Std":>10}  {"F1_Hard":>10}  {"ΔF1 (drop)":>14}  {"Drop%":>12}'
print(hdr); print('-' * 80)
for _, row in temporal_summary.iterrows():
    print(f'{row["model_name"]:18s}  '
          f'{row["f1_standard_mean"]:.4f}±{row["f1_standard_std"]:.4f}  '
          f'{row["f1_hard_mean"]:.4f}±{row["f1_hard_std"]:.4f}  '
          f'{row["drop_abs_mean"]:+.4f}±{row["drop_abs_std"]:.4f}  '
          f'{row["drop_pct_mean"]:+.2f}%±{row["drop_pct_std"]:.2f}%')

print('\n── E2: Feature Perturbation Slope ─────────────────────────────────────────────────')
hdr2 = f'{"Classifier":18s}  {"F1@σ=0":>10}  {"F1@σ=0.05":>12}  {"F1@σ=0.2":>11}  {"Slope (mean±std)":>18}'
print(hdr2); print('-' * 80)
for clf_name in MODEL_ORDER:
    d  = perturbation_summary[perturbation_summary['model_name'] == clf_name].sort_values('sigma')
    s0 = d[d['sigma'] == 0.0]
    s5 = d[d['sigma'] == 0.05]
    s2 = d[d['sigma'] == 0.2]
    sl = slope_summary[slope_summary['model_name'] == clf_name]
    f = lambda r: f'{r["f1_macro_mean"].values[0]:.4f}±{r["f1_macro_std"].values[0]:.4f}'
    print(f'{clf_name:18s}  {f(s0):>10}  {f(s5):>12}  {f(s2):>11}  '
          f'{sl["slope_mean"].values[0]:+.4f}±{sl["slope_std"].values[0]:.4f}')

print('\n── E3: Class Prior Shift ───────────────────────────────────────────────────────────')
col_labels = '  '.join(f'{s.replace(chr(10)," "):>22}' for s in set_names)
hdr3 = f'{"Classifier":18s}  {col_labels}  {"Mean F1":>10}  {"Std(dists)":>11}'
print(hdr3); print('-' * 100)
for i, clf_name in enumerate(MODEL_ORDER):
    vals = '  '.join(
        f'{f1_matrix_mu[i,j]:.4f}±{f1_matrix_sd[i,j]:.4f}'.rjust(22)
        for j in range(len(set_names))
    )
    print(f'{clf_name:18s}  {vals}  '
          f'{e3_mean_f1[i]:.4f}  {e3_std_f1[i]:.4f}')

print('\n── McNemar (QSVM vs baselines, E1 hard set, mean over runs) ───────────────────────')
print(f'{"Baseline":22s}  {"chi2_mean":>10}  {"chi2_std":>9}  {"p_mean":>9}  {"p_std":>9}  Sig')
print('-' * 75)
for _, row in mcnemar_summary.iterrows():
    sig = '**' if row['p_value_mean'] < 0.05 else '(ns)'
    print(f'{row["baseline"]:22s}  {row["chi2_mean"]:>10.3f}  {row["chi2_std"]:>9.3f}  '
          f'{row["p_value_mean"]:>9.4f}  {row["p_value_std"]:>9.4f}  {sig}')

print('\n' + '=' * 90)
print('KẾT LUẬN C4 MULTI-RUN:')
print('  - E1: QSVM F1-drop mean ± std vs. SVM-RBF (xem bảng trên)')
print('  - E2: QSVM degradation slope nhỏ hơn ở σ ≤ 0.05 (flatter curve = more robust)')
print('  - E3: QSVM std_across_distributions nhỏ hơn (stable hơn khi prior shift)')
print('=' * 90)